# Movie Ratings Analysis - Normalization and Standardization

In this notebook I will load a CSV file with movie ratings from friends, compute some averages, and then try normalizing and standardizing the data to see how that changes things.

## Step 0 - Imports

I'm importing pandas for the dataframe work and pathlib to handle the file path so it doesn't break depending on where I run the notebook from.

In [ ]:
import pandas as pd
from pathlib import Path

## Step 1 - Load the CSV

A few things I needed to figure out here:
- index_col=0 tells pandas to use the first column (the names) as the row index instead of treating it as data
- encoding='utf-8-sig' strips a hidden character that Excel sometimes puts at the start of CSV files
- Path.cwd() builds the full path to the file so it works no matter where the notebook is opened from, as long as the CSV is in the same folder

In [ ]:
CSV_PATH = Path.cwd() / "Movie_Ratings_by_Friends.csv"

df = pd.read_csv(CSV_PATH, index_col=0, encoding="utf-8-sig")
print("Shape:", df.shape)
df

## Step 2 - Average ratings (original)

I'm using .mean() here. By default pandas skips NaN values when calculating the mean, which is what we want since not everyone rated every movie.

- axis=1 averages across columns, so we get one average per user
- axis=0 averages across rows, so we get one average per movie

In [ ]:
print("Average rating per user (original):")
print(df.mean(axis=1).round(2).to_frame(name="Avg Rating"))

print("\nAverage rating per movie (original):")
print(df.mean(axis=0).round(2).to_frame(name="Avg Rating"))

## Step 3 - Normalized ratings

Min-max normalization rescales each user's ratings so their lowest becomes 0 and their highest becomes 1. The formula is:

$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

I'm doing this per user (per row) because different people use the rating scale differently. Someone who never gives above a 3 shouldn't be directly compared to someone who gives 4s and 5s all the time.

I also added a check for when min equals max (e.g. if someone only rated one movie), because that would cause a division by zero error.

In [ ]:
def normalize_row(row):
    row_min = row.min()
    row_max = row.max()
    if row_max == row_min:  # avoid division by zero
        return row.apply(lambda x: 0.5 if pd.notna(x) else float("nan"))
    return (row - row_min) / (row_max - row_min)

# apply with axis=1 runs the function on each row (each user)
df_normalized = df.apply(normalize_row, axis=1)

print("Normalized ratings (0 to 1 per user):")
df_normalized.round(4)

In [ ]:
print("Average rating per user (normalized):")
print(df_normalized.mean(axis=1).round(4).to_frame(name="Avg Normalized"))

print("\nAverage rating per movie (normalized):")
print(df_normalized.mean(axis=0).round(4).to_frame(name="Avg Normalized"))

## Step 4 - Normalized vs Actual Ratings: Pros and Cons

Advantages of using normalized ratings:
- It removes the personal bias each user has in how they use the rating scale. A 4 from someone who never gives 5s is very different from a 4 from someone who gives 5s all the time.
- It makes it easier to compare relative preferences across users.
- Many machine learning algorithms expect input values in the 0 to 1 range, so normalization helps with that.

Disadvantages of using normalized ratings:
- We lose the actual rating values. A normalized score of 1.0 could mean a 3 or a 5 depending on the user, so we can't tell how much someone actually liked something.
- It is sensitive to outliers. One very high or very low rating changes the scale for all the other ratings that user gave.
- When a user has a lot of missing ratings, the min and max are based on fewer values, which makes the normalization less reliable.
- The per-movie averages are harder to explain to someone who doesn't know what normalization means.

## Step 5 (Extra Credit) - Standardized ratings

Standardization (also called Z-score scaling) works differently from normalization. Instead of fitting everything into a 0 to 1 range, it shifts each user's ratings so their mean is 0 and their standard deviation is 1. The formula is:

$$x_{std} = \frac{x - \bar{x}}{\sigma}$$

A positive Z-score means the rating was above that user's average, and a negative one means it was below.

The main difference from normalization is that standardization doesn't have a fixed output range, which makes it more robust to outliers. It is also commonly used in machine learning when the data is expected to follow a normal distribution.

Again I need to handle the edge case where std is 0 (only one rating or all ratings are the same).

In [ ]:
def standardize_row(row):
    row_mean = row.mean()
    row_std = row.std()
    if pd.isna(row_std) or row_std == 0:  # avoid division by zero
        return row.apply(lambda x: 0.0 if pd.notna(x) else float("nan"))
    return (row - row_mean) / row_std

df_standardized = df.apply(standardize_row, axis=1)

print("Standardized ratings (Z-score per user):")
df_standardized.round(4)

In [ ]:
print("Average rating per user (standardized):")
print(df_standardized.mean(axis=1).round(4).to_frame(name="Avg Z-score"))

print("\nAverage rating per movie (standardized):")
print(df_standardized.mean(axis=0).round(4).to_frame(name="Avg Z-score"))

Something interesting to notice: every user's average Z-score comes out to exactly 0.0. That is not a coincidence, it is guaranteed by the math since we are subtracting each person's own mean. So standardization always centers each user's ratings at zero.

For the per-movie averages, a positive number means that movie was generally rated above each person's personal average, and a negative number means the opposite.